# Resolución de Sudoku mediante backtracking

Este notebook explica paso a paso cómo funciona el algoritmo utilizado en el proyecto **Sudoku Resolver**.

El objetivo es comprender cómo se representa el tablero, cómo se validan los números y cómo el algoritmo de **backtracking** prueba alternativas hasta encontrar una solución.

## 1. Preparación del entorno

El notebook importa las funciones directamente desde `src/solver.py`. De esta forma, los ejemplos utilizan exactamente el mismo código que ejecutan la aplicación de Pygame y los tests.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

ruta_actual = Path.cwd().resolve()
raiz_proyecto = (
    ruta_actual.parent
    if ruta_actual.name == "notebooks"
    else ruta_actual
)

if not (raiz_proyecto / "src").exists():
    raise FileNotFoundError(
        "No se encuentra la carpeta src. Abre el notebook desde la raíz del proyecto."
    )

if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

from src.solver import (
    TABLERO_INICIAL,
    buscar_casilla_vacia,
    mostrar_tablero,
    numero_es_valido,
    numero_valido_en_bloque,
    numero_valido_en_columna,
    numero_valido_en_fila,
    resolver,
    tablero_es_valido,
)

print(f"Proyecto: {raiz_proyecto}")
print(f"Python: {sys.executable}")

## 2. Representación del tablero

El Sudoku se representa mediante una lista con nueve listas internas, una por cada fila.

- Los números del `1` al `9` representan casillas ocupadas.
- El valor `0` representa una casilla vacía.

Creamos una copia para no modificar `TABLERO_INICIAL`, ya que el resolutor escribe los números sobre el tablero recibido.

In [ ]:
tablero = [fila.copy() for fila in TABLERO_INICIAL]
mostrar_tablero(tablero)

## 3. Búsqueda de una casilla vacía

`buscar_casilla_vacia` recorre el tablero de izquierda a derecha y de arriba abajo. Devuelve una tupla `(fila, columna)` con la primera posición cuyo valor sea `0`.

Si no quedan casillas vacías, devuelve `None`. Esta situación será el caso base que indique que el Sudoku está completo.

In [ ]:
primera_casilla_vacia = buscar_casilla_vacia(tablero)
print(f"Primera casilla vacía: {primera_casilla_vacia}")

El resultado esperado es `(0, 2)`: la primera casilla vacía está en la fila `0` y la columna `2`. Python comienza a numerar las posiciones desde cero.

## 4. Validación de números

Antes de colocar un número, el programa comprueba que no se encuentre repetido en la misma fila, en la misma columna ni en el mismo bloque de 3 × 3.

### 4.1. Validación de la fila y la columna

Las siguientes comprobaciones muestran un caso permitido y otro no permitido para cada función.

In [ ]:
print("Fila: número 4 permitido ->", numero_valido_en_fila(tablero, 4, 0))
print("Fila: número 5 permitido ->", numero_valido_en_fila(tablero, 5, 0))

print("Columna: número 4 permitido ->", numero_valido_en_columna(tablero, 4, 2))
print("Columna: número 8 permitido ->", numero_valido_en_columna(tablero, 8, 2))

### 4.2. Validación del bloque de 3 × 3

Cada casilla pertenece a uno de los nueve bloques del Sudoku. Para localizar el inicio del bloque se utiliza división entera:

```python
inicio_fila = (fila // 3) * 3
inicio_columna = (columna // 3) * 3
```

Por ejemplo, la casilla situada en la fila `4` y la columna `5` pertenece al bloque que comienza en la fila `3` y la columna `3`.

In [ ]:
print("Bloque: número 4 permitido ->", numero_valido_en_bloque(tablero, 4, 0, 2))
print("Bloque: número 9 permitido ->", numero_valido_en_bloque(tablero, 9, 0, 2))

### 4.3. Validación completa de una posición

Las comprobaciones de fila, columna y bloque se combinan en `numero_es_valido`. El resultado solo es `True` cuando las tres condiciones se cumplen al mismo tiempo.

In [ ]:
print("¿Se puede colocar 4 en (0, 2)?", numero_es_valido(tablero, 4, 0, 2))
print("¿Se puede colocar 5 en (0, 2)?", numero_es_valido(tablero, 5, 0, 2))
print("¿Se puede colocar 6 en (0, 2)?", numero_es_valido(tablero, 6, 0, 2))

## 5. Algoritmo de backtracking

El backtracking construye la solución de manera incremental:

1. Busca la primera casilla vacía.
2. Prueba los números del `1` al `9`.
3. Si un número es válido, lo coloca temporalmente.
4. Continúa resolviendo el resto del tablero mediante una llamada recursiva.
5. Si llega a un punto sin opciones válidas, borra el último número colocado y prueba otra alternativa.
6. Si no quedan casillas vacías, devuelve `True` porque el Sudoku está resuelto.

Ese paso de borrar un número y regresar a una decisión anterior es lo que da nombre al algoritmo.

### Esquema del algoritmo

```text
buscar una casilla vacía

si no existe:
    el Sudoku está resuelto

para cada número del 1 al 9:
    si el número es válido:
        colocarlo
        intentar resolver el resto

        si el intento falla:
            borrar el número y probar el siguiente

si ninguno funciona:
    volver a la decisión anterior
```

## 6. Resolución del Sudoku

La función pública `resolver` valida primero la estructura y las reglas del tablero. Después aplica el backtracking y modifica la copia del tablero directamente.

In [ ]:
tablero_resuelto = [fila.copy() for fila in TABLERO_INICIAL]

inicio = perf_counter()
tiene_solucion = resolver(tablero_resuelto)
tiempo = perf_counter() - inicio

print(f"¿Se encontró una solución? {tiene_solucion}")
print(f"Tiempo empleado: {tiempo:.6f} segundos")
print()
mostrar_tablero(tablero_resuelto)

## 7. Comprobación automática del resultado

Además de observar el tablero, comprobamos que la función encontró una solución, que ya no contiene ceros y que todas las reglas del Sudoku siguen cumpliéndose.

In [ ]:
assert tiene_solucion
assert buscar_casilla_vacia(tablero_resuelto) is None
assert tablero_es_valido(tablero_resuelto)

print("Comprobación superada: el tablero está completo y es válido.")

## 8. Ejemplo de tablero inválido

El resolutor también rechaza tableros que ya incumplen las reglas. En este ejemplo introducimos dos números `5` en la primera fila.

In [ ]:
tablero_invalido = [fila.copy() for fila in TABLERO_INICIAL]
tablero_invalido[0][2] = 5

print("¿El tablero inicial es válido?", tablero_es_valido(TABLERO_INICIAL))
print("¿El tablero modificado es válido?", tablero_es_valido(tablero_invalido))
print("¿Puede resolverlo?", resolver(tablero_invalido))

## 9. Conclusiones

En este proyecto se han aplicado los siguientes conceptos:

- Representación de datos mediante listas bidimensionales.
- División del problema en funciones pequeñas y reutilizables.
- Validación de filas, columnas y bloques de 3 × 3.
- Recursividad y backtracking.
- Protección del tablero original mediante copias.
- Validación de entradas y comprobaciones automáticas.
- Separación entre la lógica del resolutor y la interfaz gráfica de Pygame.

El backtracking no prueba todas las combinaciones de manera indiscriminada: abandona una rama tan pronto como detecta que una elección no puede conducir a una solución válida.